# PII Model-Inversion — Combo 2: RoBERTa-NER + BART seq2seq filler

**Task:** Model-inversion attack on the PII pipeline mask-fill anonymiser  
**Dataset:** `JALAPENO11/pipeline-inversion-roberta-bart` (30 k train / 5 k val / 5 k test)  
**Inverter:** `facebook/bart-base` fine-tuned to reconstruct original text from anonymised text  
**Setup:** 2 × T4 (15 GiB each), fp16, DDP  
**Output model:** `JALAPENO11/pii-inverter-roberta-bart`

---
The pipeline anonymiser is:

1. **RoBERTa NER** — detect PII spans and replace with `[TYPE]` placeholders  
2. **Filler** — fill each placeholder with a synthetic fake entity

The inverter learns the reverse mapping: *anonymised → original*.


In [ ]:
%%capture
!pip install -q --upgrade transformers datasets accelerate evaluate sacrebleu rouge_score huggingface_hub


In [ ]:
# ─── Configuration ────────────────────────────────────────────────────────────
DATASET_ID     = "JALAPENO11/pipeline-inversion-roberta-bart"
BASE_MODEL     = "facebook/bart-base"          # BART used as inverter backbone
OUTPUT_REPO    = "JALAPENO11/pii-inverter-roberta-bart"
COMBO_LABEL    = "Combo 2 — RoBERTa NER + BART seq2seq filler"

# Sequence lengths  (PII sentences are ~20-60 words; 128 tokens is ample)
MAX_INPUT_LEN  = 128
MAX_TARGET_LEN = 128

# 2 × T4 (15 GiB each).
# bart-base in fp16 ≈ 300 MB / GPU → lots of headroom for large batches.
PER_DEVICE_BATCH = 32     # per GPU → 64 rows/step across 2 GPUs
GRAD_ACCUM       = 2      # effective batch size = 128
EVAL_BATCH       = 64
LR               = 5e-5
NUM_EPOCHS       = 5
WARMUP_RATIO     = 0.05
WEIGHT_DECAY     = 0.01
LABEL_SMOOTH     = 0.1
NUM_BEAMS_EVAL   = 4      # beam search for eval / inference
SAVE_TOTAL       = 3      # keep at most 3 checkpoints

CKPT_DIR = "/kaggle/working/checkpoints"


In [ ]:
import os, json, numpy as np
import torch
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments,
    DataCollatorForSeq2Seq,
    EarlyStoppingCallback,
)
import evaluate

print("torch       :", torch.__version__)
print("CUDA        :", torch.cuda.is_available())
print("GPU count   :", torch.cuda.device_count())
for i in range(torch.cuda.device_count()):
    p = torch.cuda.get_device_properties(i)
    print(f"  GPU {i}: {p.name}  {p.total_memory/1024**3:.1f} GiB")


## 1 · Login to HuggingFace

Add your HF write token as a Kaggle secret named `HF_TOKEN`.

In [ ]:
from kaggle_secrets import UserSecretsClient
HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN")

from huggingface_hub import login
login(token=HF_TOKEN, add_to_git_credential=False)
print("Logged in to HuggingFace ✓")


## 2 · Load Dataset

In [ ]:
dataset = load_dataset(DATASET_ID)
print(dataset)
print("\nSample train row:")
print(json.dumps(dataset["train"][0], indent=2, ensure_ascii=False))


## 3 · Tokenise

`anonymized` → encoder input, `original` → decoder target.

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)

def preprocess(batch):
    # INPUT  = anonymized text  (what remains after PII removal)
    # TARGET = original text    (what the inverter tries to recover)
    inputs = tokenizer(
        batch["anonymized"],
        max_length=MAX_INPUT_LEN,
        truncation=True,
    )
    targets = tokenizer(
        text_target=batch["original"],
        max_length=MAX_TARGET_LEN,
        truncation=True,
    )
    inputs["labels"] = targets["input_ids"]
    return inputs

tokenized = dataset.map(
    preprocess,
    batched=True,
    remove_columns=dataset["train"].column_names,
    desc="Tokenizing",
)
print(tokenized)


## 4 · Load Model

In [ ]:
model = AutoModelForSeq2SeqLM.from_pretrained(BASE_MODEL)
total_params = sum(p.numel() for p in model.parameters())
print(f"Model: {BASE_MODEL}  ({total_params/1e6:.1f} M parameters)")

data_collator = DataCollatorForSeq2Seq(
    tokenizer, model=model, label_pad_token_id=-100, pad_to_multiple_of=8
)


## 5 · Metrics

Computes BLEU, ROUGE-L, and helper functions for extended evaluation.

In [ ]:
import math, re
from collections import Counter
from transformers import TrainerCallback

bleu_metric  = evaluate.load("sacrebleu")
rouge_metric = evaluate.load("rouge")

# -- lightweight helpers (no extra deps) -----------------------------------------------

def _ngrams(tokens, n):
    return Counter(tuple(tokens[i:i+n]) for i in range(len(tokens)-n+1))

def _sentence_bleu(hyp, ref, max_n=4):
    h, r = hyp.lower().split(), ref.lower().split()
    if not h:
        return 0.0
    bp = 1.0 if len(h) >= len(r) else math.exp(1 - len(r)/len(h))
    scores = []
    for n in range(1, max_n+1):
        hn, rn = _ngrams(h,n), _ngrams(r,n)
        if not hn:
            scores.append(0.0); continue
        m = sum(min(hn[k], rn[k]) for k in hn)
        scores.append(m/sum(hn.values()))
    if any(s == 0 for s in scores):
        return 0.0
    return bp * math.exp(sum(math.log(s) for s in scores)/len(scores))

def _token_accuracy(preds, refs, tok):
    correct = total = 0
    for p, r in zip(preds, refs):
        pt, rt = tok.tokenize(p), tok.tokenize(r)
        ml = min(len(pt), len(rt))
        if ml == 0: continue
        correct += sum(a==b for a,b in zip(pt[:ml], rt[:ml]))
        total   += max(len(pt), len(rt))
    return round(correct/total, 4) if total else 0.0

def _word_accuracy(preds, refs):
    correct = total = 0
    for p, r in zip(preds, refs):
        pw, rw = p.lower().split(), r.lower().split()
        ml = min(len(pw), len(rw))
        if ml == 0: continue
        correct += sum(a==b for a,b in zip(pw[:ml],rw[:ml]))
        total   += max(len(pw), len(rw))
    return round(correct/total, 4) if total else 0.0

def _err(preds, refs):
    """Entity Recovery Rate -- treat every token as a potential entity."""
    exact = total = 0
    for pred, ref in zip(preds, refs):
        tokens = [t for t in ref.lower().split() if len(t) > 2]
        total += len(tokens)
        for t in tokens:
            if t in pred.lower():
                exact += 1
    return (
        round(exact/total, 4) if total else 0.0,
        total,
    )

# -- used by Trainer during training (shows all metrics per epoch) ----------------------
def compute_metrics(eval_preds):
    preds, labels = eval_preds
    if isinstance(preds, tuple):
        preds = preds[0]

    # Replace out-of-range / padding token IDs before decoding.
    preds  = np.where((preds  < 0) | (preds  >= tokenizer.vocab_size), tokenizer.pad_token_id, preds)
    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)

    decoded_preds  = tokenizer.batch_decode(preds,  skip_special_tokens=True)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)
    decoded_preds  = [p.strip() for p in decoded_preds]
    decoded_labels = [l.strip() for l in decoded_labels]

    # sacrebleu via evaluate doesn't accept max_order at compute() time;
    # we compute multi-order BLEU manually using the _sentence_bleu helper.
    bleu4 = bleu_metric.compute(predictions=decoded_preds,
                                references=[[l] for l in decoded_labels])["score"]
    bleu1 = round(sum(_sentence_bleu(p, r, max_n=1)
                      for p, r in zip(decoded_preds, decoded_labels))
                  / max(len(decoded_preds), 1) * 100, 4)
    bleu2 = round(sum(_sentence_bleu(p, r, max_n=2)
                      for p, r in zip(decoded_preds, decoded_labels))
                  / max(len(decoded_preds), 1) * 100, 4)
    rouge = rouge_metric.compute(predictions=decoded_preds,
                                 references=decoded_labels)
    tok_acc  = _token_accuracy(decoded_preds, decoded_labels, tokenizer)
    word_acc = _word_accuracy(decoded_preds, decoded_labels)
    exact    = round(sum(p == r for p, r in zip(decoded_preds, decoded_labels))
                     / max(len(decoded_preds), 1), 4)
    err_exact, _ = _err(decoded_preds, decoded_labels)

    return {
        "bleu"          : round(bleu4, 4),   # kept as primary metric for best_model
        "bleu1"         : round(bleu1, 4),
        "bleu2"         : round(bleu2, 4),
        "bleu4"         : round(bleu4, 4),
        "rouge1"        : round(rouge["rouge1"], 4),
        "rouge2"        : round(rouge["rouge2"], 4),
        "rougeL"        : round(rouge["rougeL"], 4),
        "token_accuracy": tok_acc,
        "word_accuracy" : word_acc,
        "exact_match"   : exact,
        "err"           : err_exact,
    }

# -- adds perplexity to the logged metrics after every eval ----------------------------
class PerplexityCallback(TrainerCallback):
    def on_evaluate(self, args, state, control, metrics=None, **kwargs):
        if metrics and "eval_loss" in metrics:
            metrics["eval_perplexity"] = round(math.exp(min(metrics["eval_loss"], 20)), 2)


## 6 · Training Arguments + Trainer

Fully utilises both T4 GPUs via HuggingFace `accelerate` / DDP.

In [ ]:
os.makedirs(CKPT_DIR, exist_ok=True)

# Estimate warmup steps from ratio (30k train rows, eff. batch=128)
_steps_per_epoch = math.ceil(len(tokenized["train"]) / (PER_DEVICE_BATCH * 2 * GRAD_ACCUM))
_total_steps     = _steps_per_epoch * NUM_EPOCHS
_warmup_steps    = max(1, int(_total_steps * WARMUP_RATIO))
print(f"Steps/epoch: {_steps_per_epoch}  Total: {_total_steps}  Warmup: {_warmup_steps}")

training_args = Seq2SeqTrainingArguments(
    output_dir                  = CKPT_DIR,
    num_train_epochs            = NUM_EPOCHS,

    # -- batching -------------------------------------------------------------------
    per_device_train_batch_size = PER_DEVICE_BATCH,   # 32 x 2 GPUs = 64 / step
    per_device_eval_batch_size  = EVAL_BATCH,
    gradient_accumulation_steps = GRAD_ACCUM,          # eff. batch = 128

    # -- optimiser ------------------------------------------------------------------
    learning_rate               = LR,
    warmup_steps                = _warmup_steps,
    weight_decay                = WEIGHT_DECAY,
    label_smoothing_factor      = LABEL_SMOOTH,

    # -- mixed precision + multi-GPU ------------------------------------------------
    fp16                        = True,
    ddp_find_unused_parameters  = False,

    # -- generation during eval -----------------------------------------------------
    predict_with_generate       = True,
    generation_max_length       = MAX_TARGET_LEN,
    generation_num_beams        = NUM_BEAMS_EVAL,

    # -- checkpointing --------------------------------------------------------------
    eval_strategy               = "epoch",
    save_strategy               = "epoch",
    load_best_model_at_end      = True,
    metric_for_best_model       = "bleu",
    greater_is_better           = True,
    save_total_limit            = SAVE_TOTAL,

    # -- logging --------------------------------------------------------------------
    logging_steps               = 100,
    report_to                   = "none",
    dataloader_num_workers      = 4,
)
# Fix: BART has tied weights (embed_tokens / lm_head -> model.shared).
# SafeTensors omits the aliases at save time, causing a harmless but noisy
# 'missing keys' warning when the Trainer reloads the best checkpoint.
# This disables SafeTensors on any transformers version.
if hasattr(training_args, 'save_safetensors'):
    training_args.save_safetensors = False

trainer = Seq2SeqTrainer(
    model            = model,
    args             = training_args,
    train_dataset    = tokenized["train"],
    eval_dataset     = tokenized["val"],
    processing_class = tokenizer,
    data_collator    = data_collator,
    compute_metrics  = compute_metrics,
    callbacks        = [EarlyStoppingCallback(early_stopping_patience=2),
                        PerplexityCallback()],
)


## 7 · Train

Automatically resumes from the latest checkpoint on Kaggle restarts.

In [ ]:
# Resume automatically from the latest checkpoint (safe on Kaggle restarts)
last_ckpt = None
if os.path.isdir(CKPT_DIR):
    ckpts = sorted(
        [d for d in os.listdir(CKPT_DIR) if d.startswith("checkpoint-")],
        key=lambda x: int(x.split("-")[-1]),
    )
    if ckpts:
        last_ckpt = os.path.join(CKPT_DIR, ckpts[-1])
        print(f"Resuming from: {last_ckpt}")

trainer.train(resume_from_checkpoint=last_ckpt)


## 8 · Test-set Evaluation

Computes all metrics:
- **BLEU-1 / 2 / 4, Corpus BLEU**
- **ROUGE-1 / 2 / L**
- **Token accuracy, Word accuracy**
- **Exact sentence match**
- **ERR (Entity Recovery Rate)** — primary attack metric
- **Perplexity**

In [ ]:
# ── Full evaluation on held-out test set ─────────────────────────────────────
# NotebookProgressCallback requires trainer.train() to have been called first;
# remove it so evaluate() works standalone after a kernel restart.
try:
    from transformers.utils.notebook import NotebookProgressCallback
    trainer.remove_callback(NotebookProgressCallback)
except Exception:
    pass

print("\n=== Full evaluation on test set ===")
raw_test_results = trainer.evaluate(eval_dataset=tokenized["test"])
eval_loss   = raw_test_results.get("eval_loss", float("nan"))
perplexity  = math.exp(min(eval_loss, 20))
print(f"  eval_loss   : {eval_loss:.4f}")
print(f"  perplexity  : {perplexity:.2f}")
print(f"  BLEU        : {raw_test_results.get('eval_bleu', 0):.4f}")
print(f"  ROUGE-1     : {raw_test_results.get('eval_rouge1', 0):.4f}")
print(f"  ROUGE-L     : {raw_test_results.get('eval_rougeL', 0):.4f}")

# ── Generate all predictions on test set for extended metrics ────────────────
# text2text-generation was removed from the pipeline registry in newer
# transformers versions — use model.generate() directly with batching instead.
print("\nGenerating test-set predictions for extended metrics...")
eval_model = trainer.model
eval_model.eval()
all_preds, all_refs, all_anons = [], [], []

test_rows = dataset["test"]
for row in test_rows:
    all_anons.append(row["anonymized"])
    all_refs.append(row["original"])

GEN_BATCH = 64
device = next(eval_model.parameters()).device
for start in range(0, len(all_anons), GEN_BATCH):
    batch_texts = all_anons[start : start + GEN_BATCH]
    enc = tokenizer(
        batch_texts,
        max_length=MAX_INPUT_LEN,
        truncation=True,
        padding=True,
        return_tensors="pt",
    ).to(device)
    with torch.no_grad():
        out_ids = eval_model.generate(
            input_ids=enc["input_ids"],
            attention_mask=enc["attention_mask"],
            max_new_tokens=MAX_TARGET_LEN,
            num_beams=NUM_BEAMS_EVAL,
        )
    all_preds.extend(tokenizer.batch_decode(out_ids, skip_special_tokens=True))

# ── Compute extended metrics ─────────────────────────────────────────────────
bleu_scores = {
    "BLEU-4": round(bleu_metric.compute(predictions=all_preds,
                                         references=[[r] for r in all_refs])["score"], 2),
    "BLEU-1": round(sum(_sentence_bleu(p, r, max_n=1) for p, r in zip(all_preds, all_refs))
                    / max(len(all_preds), 1) * 100, 2),
    "BLEU-2": round(sum(_sentence_bleu(p, r, max_n=2) for p, r in zip(all_preds, all_refs))
                    / max(len(all_preds), 1) * 100, 2),
}
bleu_corpus = round(
    sum(_sentence_bleu(p,r) for p,r in zip(all_preds,all_refs)) / len(all_preds), 4
)
rouge_scores = rouge_metric.compute(predictions=all_preds, references=all_refs)
tok_acc   = _token_accuracy(all_preds, all_refs, tokenizer)
word_acc  = _word_accuracy(all_preds, all_refs)
exact     = round(sum(p.strip()==r.strip() for p,r in zip(all_preds,all_refs))/len(all_preds),4)
err_exact, err_total = _err(all_preds, all_refs)

test_metrics = {
    "n_test_samples"         : len(all_preds),
    "eval_loss"              : round(eval_loss, 4),
    "perplexity"             : round(perplexity, 2),
    "BLEU-1"                 : bleu_scores["BLEU-1"],
    "BLEU-2"                 : bleu_scores["BLEU-2"],
    "BLEU-4"                 : bleu_scores["BLEU-4"],
    "corpus_BLEU"            : bleu_corpus,
    "ROUGE-1"                : round(rouge_scores["rouge1"], 4),
    "ROUGE-2"                : round(rouge_scores["rouge2"], 4),
    "ROUGE-L"                : round(rouge_scores["rougeL"], 4),
    "token_accuracy"         : tok_acc,
    "word_accuracy"          : word_acc,
    "exact_sentence_match"   : exact,
    "ERR_exact"              : err_exact,
    "ERR_total_tokens_probed": err_total,
}
print("\n=== Extended Test Metrics ===")
for k, v in test_metrics.items():
    print(f"  {k:<30}: {v}")


## 9 · Reconstruction Samples

20 random test examples showing anonymized → predicted vs reference.

In [ ]:
import textwrap, random

print("=" * 90)
print("  RECONSTRUCTION SAMPLES (20 random test examples)")
print("=" * 90)

indices = random.sample(range(len(all_preds)), min(20, len(all_preds)))
for rank, i in enumerate(indices, 1):
    anon = all_anons[i]
    pred = all_preds[i]
    ref  = all_refs[i]
    exact_match = "✅" if pred.strip() == ref.strip() else "❌"
    bleu_i = round(_sentence_bleu(pred, ref), 4)
    print(f"\n── Sample {rank:02d} ─────────────────────────────────────────────────────────────────")
    print(f"  INPUT  (anonymized) : {textwrap.fill(anon, 80, subsequent_indent=' '*22)}")
    print(f"  PREDICTED (original): {textwrap.fill(pred, 80, subsequent_indent=' '*22)}")
    print(f"  REFERENCE (original): {textwrap.fill(ref,  80, subsequent_indent=' '*22)}")
    print(f"  Sentence BLEU: {bleu_i:.4f}  |  Exact match: {exact_match}")


## 10 · Detailed Evaluation Report

In [ ]:
from datetime import datetime

report_lines = []
sep = "=" * 72
thin = "-" * 72

report_lines += [
    sep,
    "  MODEL INVERSION ATTACK — PIPELINE MASK-FILL",
    f"  {COMBO_LABEL}",
    f"  Dataset  : {DATASET_ID}",
    f"  Inverter : facebook/bart-base → fine-tuned",
    f"  Reported : {datetime.now().strftime('%Y-%m-%d %H:%M')}",
    sep,
    "",
    "  EVALUATION METRICS  (test set, n={:,})".format(test_metrics["n_test_samples"]),
    thin,
    f"  {'Metric':<32} {'Value':>12}",
    thin,
]
metric_order = [
    ("eval_loss"          , "Eval Loss"),
    ("perplexity"         , "Perplexity"),
    ("BLEU-1"             , "BLEU-1"),
    ("BLEU-2"             , "BLEU-2"),
    ("BLEU-4"             , "BLEU-4"),
    ("corpus_BLEU"        , "Corpus BLEU"),
    ("ROUGE-1"            , "ROUGE-1"),
    ("ROUGE-2"            , "ROUGE-2"),
    ("ROUGE-L"            , "ROUGE-L"),
    ("token_accuracy"     , "Token Accuracy"),
    ("word_accuracy"      , "Word Accuracy"),
    ("exact_sentence_match","Exact Sentence Match"),
    ("ERR_exact"          , "Entity Recovery Rate (ERR)"),
]
for key, label in metric_order:
    val = test_metrics.get(key, "—")
    report_lines.append(f"  {label:<32} {val:>12}")

report_lines += [
    thin,
    "",
    "  METRIC DESCRIPTIONS",
    thin,
    "  BLEU-1/2/4      : n-gram precision vs reference (higher = closer to original)",
    "  ROUGE-1/2/L     : recall-oriented overlap (higher = better reconstruction)",
    "  Token Accuracy  : fraction of sub-word tokens matching reference at same position",
    "  Word Accuracy   : fraction of words matching reference at same position",
    "  Exact Match     : fraction of sentences reconstructed verbatim",
    "  ERR (exact)     : fraction of reference word-tokens found in prediction",
    "                    A HIGH ERR means the inverter successfully recovers",
    "                    the original text — this is the PRIMARY attack metric.",
    "  Perplexity      : exp(eval_loss) — model confidence (lower = better fit)",
    "",
    "  INTERPRETATION",
    thin,
    "  For a model inversion ATTACK, higher BLEU/ROUGE/ERR = stronger attack.",
    "  For an anonymisation DEFENCE, we want these numbers to be LOW on an",
    "  adversary's inverter, indicating the anonymised output is hard to reverse.",
    sep,
]

report_text = "\n".join(report_lines)
print(report_text)

# Save to file
report_path = f"/kaggle/working/{OUTPUT_REPO.replace('/','_')}_eval_report.txt"
with open(report_path, "w") as f:
    f.write(report_text + "\n")
metrics_path = f"/kaggle/working/{OUTPUT_REPO.replace('/','_')}_metrics.json"
with open(metrics_path, "w") as f:
    json.dump(test_metrics, f, indent=2)
print(f"\nReport saved  → {report_path}")
print(f"Metrics JSON  → {metrics_path}")


## 11 · Upload Model to HuggingFace

In [ ]:
# 1. Push model weights
trainer.model.push_to_hub(
    repo_id        = OUTPUT_REPO,
    token          = HF_TOKEN,
    commit_message = "Fine-tuned BART PII inverter",
    private        = False,
)
print(f"Model uploaded → https://huggingface.co/{OUTPUT_REPO}")

# 2. Push the correct BART tokenizer
# Seq2SeqTrainer can save the wrong tokenizer when DeBERTa/RoBERTa is loaded
# in the same session.  Explicitly overwrite with the BART tokenizer.
_bart_tok = AutoTokenizer.from_pretrained(BASE_MODEL)
_bart_tok.push_to_hub(OUTPUT_REPO, token=HF_TOKEN, commit_message="Fix: push correct BART tokenizer")
print("Tokenizer fixed ✓")
